# 힐 클라이밍 - MountainCarContinuous-v0 (Colab + Gradio)

**Google Colab**에서 업로드 후 실행 가능합니다.
- 연속 행동: 엔진 추력 [-1, 1]
- 선형 정책: `action = clip(w @ state, -1, 1)`
- 파라미터 w를 무작위 섭동으로 최적화 (평균 리턴이 나아지면 수용)

## 1. 라이브러리 설치

In [ ]:
!pip install -q gymnasium[classic_control] gradio matplotlib

## 2. 보상 shaping, 정책 및 학습 코드

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pkg_resources is deprecated.*")

import gymnasium as gym
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import gradio as gr


class PositionRewardShaping(gym.Wrapper):
    def __init__(self, env, scale=0.5):
        super().__init__(env)
        self.scale = scale

    def step(self, action):
        obs, reward, term, trunc, info = self.env.step(action)
        position = float(obs[0])
        reward = reward + self.scale * position
        return obs, reward, term, trunc, info


class LinearPolicyMountainCar:
    def __init__(self, obs_dim, action_dim, seed=None, init_scale=0.2):
        self.obs_dim = obs_dim
        self.action_dim = action_dim
        rng = np.random.default_rng(seed)
        self.w = rng.standard_normal(obs_dim + 1) * init_scale

    def get_action(self, state, deterministic=True):
        state_b = np.concatenate([[1.0], np.asarray(state, dtype=np.float64)])
        out = np.clip(float(self.w @ state_b), -1.0, 1.0)
        return np.array([out], dtype=np.float32)

    def set_params(self, w):
        self.w = np.asarray(w, dtype=np.float64).ravel()

    def get_params(self):
        return self.w.copy()


def evaluate_policy(env, policy, n_episodes=100):
    rewards = []
    for _ in range(n_episodes):
        state, _ = env.reset()
        total = 0
        while True:
            action = policy.get_action(state, deterministic=True)
            state, reward, term, trunc, _ = env.step(action)
            total += reward
            if term or trunc:
                break
        rewards.append(total)
    return sum(rewards) / len(rewards)


def train_hill_climbing(env, policy, n_iterations=800, n_evals=30, noise_scale=0.25, noise_decay=0.997):
    best_w = policy.get_params()
    best_return = evaluate_policy(env, policy, n_episodes=n_evals)
    eval_returns = [best_return]

    for it in range(n_iterations):
        candidate_w = best_w + noise_scale * np.random.standard_normal(best_w.shape)
        policy.set_params(candidate_w)
        candidate_return = evaluate_policy(env, policy, n_episodes=n_evals)
        eval_returns.append(candidate_return)

        if candidate_return >= best_return:
            best_w = candidate_w.copy()
            best_return = candidate_return
            policy.set_params(best_w)

        noise_scale *= noise_decay

    policy.set_params(best_w)
    return eval_returns


def run_mountaincar():
    env = PositionRewardShaping(gym.make("MountainCarContinuous-v0"), scale=0.5)
    obs_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]
    policy = LinearPolicyMountainCar(obs_dim, action_dim)

    mean_before = evaluate_policy(env, policy, n_episodes=100)
    eval_returns = train_hill_climbing(env, policy, n_iterations=800, n_evals=30, noise_scale=0.25, noise_decay=0.997)
    mean_after = evaluate_policy(env, policy, n_episodes=100)
    env.close()

    env_raw = gym.make("MountainCarContinuous-v0")
    mean_raw = evaluate_policy(env_raw, policy, n_episodes=100)
    env_raw.close()

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(eval_returns, alpha=0.4, color="blue", label="Eval Return (per iter)")
    w = min(50, len(eval_returns) // 2)
    if w >= 2:
        ma = np.convolve(eval_returns, np.ones(w) / w, mode="valid")
        ax.plot(range(w - 1, len(eval_returns)), ma, color="red", label=f"Moving Avg ({w})")
    ax.set_title("Hill Climbing - MountainCarContinuous-v0 (Position Reward Shaping)")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Mean Return")
    ax.legend()
    ax.grid(True)
    plt.tight_layout()

    txt = f"""학습 전 평균 리워드 (100회, shaped): {mean_before:.1f}
학습 후 평균 리워드 (100회, shaped): {mean_after:.1f}
개선: {mean_after - mean_before:+.1f}

원본 환경 평균 리워드 (100회): {mean_raw:.1f} (목표 도달 시 +100)"""
    return fig, txt

## 3. Gradio 앱 실행

### 하이퍼파라미터 안내 (추후 슬라이더 연결 예정)

| 파라미터 | 설명 |
|----------|------|
| **n_iterations** | 전체 학습 과정을 몇 번 반복할지 결정하는 총 루프 횟수 |
| **noise_scale** | 현재 최적의 가중치에 더할 무작위 소음(Noise)의 초기 크기로 탐색 범위를 결정 |
| **noise_decay** | 학습이 진행될수록 noise_scale을 점진적으로 줄여 정밀한 최적화를 돕는 감쇠율 |
| **init_scale** | 신경망의 가중치를 처음에 얼마나 큰 범위의 무작위 값으로 설정할지 결정하는 초기화 척도 |

In [ ]:
SLIDER_GUIDANCE = """
### 하이퍼파라미터 안내 (추후 슬라이더 연결 예정)

| 파라미터 | 설명 |
|----------|------|
| **n_iterations** | 전체 학습 과정을 몇 번 반복할지 결정하는 총 루프 횟수 |
| **noise_scale** | 현재 최적의 가중치에 더할 무작위 소음(Noise)의 초기 크기로 탐색 범위를 결정 |
| **noise_decay** | 학습이 진행될수록 noise_scale을 점진적으로 줄여 정밀한 최적화를 돕는 감쇠율 |
| **init_scale** | 신경망의 가중치를 처음에 얼마나 큰 범위의 무작위 값으로 설정할지 결정하는 초기화 척도 |
"""

with gr.Blocks(title="힐 클라이밍 - MountainCar") as demo:
    gr.Markdown("# 힐 클라이밍 - MountainCarContinuous-v0")
    gr.Markdown("선형 정책 + 힐 클라이밍. 산 정상(+100)에 도달하면 성공.")
    gr.Markdown(SLIDER_GUIDANCE)

    run_btn = gr.Button("학습 및 결과 출력", variant="primary")
    plot_out = gr.Plot(label="학습 곡선")
    text_out = gr.Textbox(label="결과", lines=7)

    run_btn.click(fn=run_mountaincar, outputs=[plot_out, text_out])

demo.launch(share=False)  # share=True 로 외부 링크 생성 가능